# Model input validation

Select features to be included in the next steps, and review weekly feature coverage for those features.

Based on the case-level output from notebook 3 (`train_base_engineered.parquet`), this notebook outputs `model_feature_eligibility.csv`, which summarizes which features are included or excluded.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

import duckdb
import pandas as pd

processed_dir = Path("dataset/processed")
if not processed_dir.is_dir():
    processed_dir = Path("../dataset/processed")

engineered_path = processed_dir / "train_base_engineered.parquet"
if not engineered_path.is_file():
    raise FileNotFoundError(f"Run notebook 3 to create {engineered_path.resolve()}")

con = duckdb.connect(config={"threads": 2})
engineered = con.read_parquet(str(engineered_path))
schema = {name: str(dtype) for name, dtype in zip(engineered.columns, engineered.types)}

## Define model inputs

Keep `case_id` as the row key and `date_decision`, `WEEK_NUM` and `MONTH` as time metadata.
Use `WEEK_NUM` for chronological splits and keep `target` separately as the label.

Features labeled as `eligible` are features that will be used in the next stage (i.e. feature selection, model training, etc.) 

Fields awaiting review have role `review`, we can add features with this label later (for future improvement).

In [2]:
metadata = {
    "case_id": "Case key for joining and tracing predictions",
    "date_decision": "Decision date for chronological validation and audit",
    "WEEK_NUM": "Chronological split and weekly evaluation group",
    "MONTH": "Calendar grouping for audit",
}

eligible = {
    "person_1_applicant_mainoccupationinc_384A": "Applicant's reported main income",
    "person_1_applicant_incometype_1044T": "Applicant's reported income type",
    "person_1_applicant_empl_industry_691L": "Applicant's employment industry",
    "employment_tenure_days": "Tenure calculated from the applicant's employment start",
    "static_0_annuity_780A": "Recorded annuity. Scoring-time availability is unverified.",
    "static_0_credamount_770A": "Recorded credit amount. Scoring-time availability is unverified.",
    "annuity_to_reported_income": "Depends on whether the recorded annuity was known at scoring",
    "credit_amount_to_reported_income": "Depends on whether the recorded credit amount was known at scoring",
}
history_date_reasons = {
    "debit_card_opening": "Opening history dated before the decision",
    "deposit_opening": "Opening history dated before the decision",
    "prior_application": "Event date passes the cutoff. Availability is unverified.",
    "active_bureau_update": "Event date passes the cutoff. Availability is unverified.",
}
for prefix, reason in history_date_reasons.items():
    for suffix in ("earliest_days_ago", "latest_days_ago", "span_days"):
        eligible[f"{prefix}_{suffix}"] = reason

required = set(metadata) | {"target"} | set(eligible)
missing = sorted(required - schema.keys())
if missing:
    raise ValueError(f"Missing expected columns from notebook 3: {', '.join(missing)}")

rows = []
for name, dtype in schema.items():
    if name in metadata:
        role, reason = "metadata", metadata[name]
    elif name == "target":
        role, reason = "label", "Observed default outcome"
    elif dtype == "DATE" or dtype.startswith("TIMESTAMP"):
        role, reason = "excluded", "Raw date retained for audit"
    elif name in eligible:
        role, reason = "eligible", eligible[name]
    else:
        role, reason = "review", "Review definition and availability at scoring"

    rows.append((name, dtype, role, reason))

feature_inventory = pd.DataFrame(rows, columns=["column_name", "data_type", "role", "reason"])
eligible_inventory = feature_inventory.loc[feature_inventory["role"].eq("eligible")]
eligible_features = eligible_inventory["column_name"].tolist()
display(feature_inventory["role"].value_counts().rename("columns"))
display(eligible_inventory)

role
review      545
excluded     87
eligible     20
metadata      4
label         1
Name: columns, dtype: int64

,column_name,data_type,role,reason
7,static_0_annuity_780A,DOUBLE,eligible,Recorded annuity. Scoring-time availability is...
45,static_0_credamount_770A,DOUBLE,eligible,Recorded credit amount. Scoring-time availabil...
605,person_1_applicant_empl_industry_691L,VARCHAR,eligible,Applicant's employment industry
612,person_1_applicant_incometype_1044T,VARCHAR,eligible,Applicant's reported income type
615,person_1_applicant_mainoccupationinc_384A,DOUBLE,eligible,Applicant's reported main income
642,annuity_to_reported_income,DOUBLE,eligible,Depends on whether the recorded annuity was kn...
643,credit_amount_to_reported_income,DOUBLE,eligible,Depends on whether the recorded credit amount ...
644,employment_tenure_days,BIGINT,eligible,Tenure calculated from the applicant's employm...
645,debit_card_opening_earliest_days_ago,BIGINT,eligible,Opening history dated before the decision
646,debit_card_opening_latest_days_ago,BIGINT,eligible,Opening history dated before the decision


## Check feature coverage by week

Coverage is the percentage of cases in a week with a non-null value for the feature.
- Zero counts as a recorded value.

In [11]:
eligible_coverage

,WEEK_NUM,case_count,column_name,non_null_cases,coverage_pct
184,0,16735,static_0_annuity_780A,16735,100.000000
185,1,18841,static_0_annuity_780A,18841,100.000000
186,2,17476,static_0_annuity_780A,17476,100.000000
187,3,16108,static_0_annuity_780A,16108,100.000000
188,4,14309,static_0_annuity_780A,14309,100.000000
...,...,...,...,...,...
59979,87,17886,active_bureau_update_span_days,6195,34.636028
59980,88,14234,active_bureau_update_span_days,5472,38.443164
59981,89,13600,active_bureau_update_span_days,5352,39.352941
59982,90,12103,active_bureau_update_span_days,3272,27.034620


In [12]:
coverage_columns = feature_inventory.loc[
    ~feature_inventory["role"].isin(["metadata", "label"]), "column_name"
]
count_expressions = ", ".join(
    f'count("{name}") AS "{name}"' for name in coverage_columns
)
weekly_counts = engineered.aggregate(
    f"WEEK_NUM, count(*) AS case_count, {count_expressions}", "WEEK_NUM"
).order("WEEK_NUM").df()

if weekly_counts.empty or weekly_counts["WEEK_NUM"].isna().any():
    raise ValueError("Weekly coverage requires a nonempty table with a WEEK_NUM for every case.")

weekly_coverage = weekly_counts.melt(
    id_vars=["WEEK_NUM", "case_count"],
    var_name="column_name",
    value_name="non_null_cases",
)
weekly_coverage["coverage_pct"] = (
    100 * weekly_coverage["non_null_cases"] / weekly_coverage["case_count"]
)
eligible_coverage = weekly_coverage.loc[weekly_coverage["column_name"].isin(eligible_features)]
coverage_summary = eligible_coverage.groupby("column_name").agg(
    min_coverage_pct=("coverage_pct", "min"),
    max_coverage_pct=("coverage_pct", "max"),
    weeks_without_values=("non_null_cases", lambda counts: counts.eq(0).sum()),
)

display(coverage_summary.sort_values("min_coverage_pct").round(2))

,min_coverage_pct,max_coverage_pct,weeks_without_values
column_name,,,
person_1_applicant_empl_industry_691L,1.13,46.63,0
active_bureau_update_earliest_days_ago,3.06,68.54,0
active_bureau_update_span_days,3.06,68.54,0
active_bureau_update_latest_days_ago,3.06,68.54,0
debit_card_opening_span_days,4.39,9.59,0
debit_card_opening_latest_days_ago,4.39,9.59,0
debit_card_opening_earliest_days_ago,4.39,9.59,0
deposit_opening_earliest_days_ago,4.43,9.68,0
deposit_opening_span_days,4.43,9.68,0


## Save feature eligibility

Notebook 5 can read this inventory and use the rows with `role == "eligible"`
to build its starting model matrix.

In [13]:
inventory_path = processed_dir / "model_feature_eligibility.csv"
with TemporaryDirectory(dir=processed_dir) as temp_dir:
    staged_path = Path(temp_dir) / inventory_path.name
    feature_inventory.to_csv(staged_path, index=False)
    staged_path.replace(inventory_path)
print(f"Saved {len(eligible_features)} eligible features to {inventory_path.resolve()}")

Saved 20 eligible features to /home/georgiusbenedikt09/MLOps_Credit_Risk/dataset/processed/model_feature_eligibility.csv


In [14]:
# Inspect the data types in the eligible feature subset.
eligible_data = engineered.select(
    *(duckdb.ColumnExpression(name) for name in eligible_features)
)
eligible_dtypes = pd.DataFrame({
    "column_name": eligible_data.columns,
    "data_type": [str(dtype) for dtype in eligible_data.types],
})
display(eligible_dtypes)

,column_name,data_type
0,static_0_annuity_780A,DOUBLE
1,static_0_credamount_770A,DOUBLE
2,person_1_applicant_empl_industry_691L,VARCHAR
3,person_1_applicant_incometype_1044T,VARCHAR
4,person_1_applicant_mainoccupationinc_384A,DOUBLE
5,annuity_to_reported_income,DOUBLE
6,credit_amount_to_reported_income,DOUBLE
7,employment_tenure_days,BIGINT
8,debit_card_opening_earliest_days_ago,BIGINT
9,debit_card_opening_latest_days_ago,BIGINT


In [24]:
eligible_data.select("person_1_applicant_empl_industry_691L").df().info()

<class 'pandas.DataFrame'>
RangeIndex: 1526659 entries, 0 to 1526658
Data columns (total 1 columns):
 #   Column                                 Non-Null Count   Dtype
---  ------                                 --------------   -----
 0   person_1_applicant_empl_industry_691L  522236 non-null  str  
dtypes: str(1)
memory usage: 14.8 MB
